# Customer Segmentation & Churn Pattern Analytics in European Banking**Unified Mentor — The European Central Bank**This notebook performs the full exploratory data analysis (EDA) and segmentation-drivenchurn analysis described in the project requirements: data validation, cleaning,derived-segment creation, churn distribution analysis, comparative demographic analysis,and high-value customer churn analysis.

## 1. Setup

In [ ]:
import syssys.path.append('..')import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom src.data_processing import load_and_prepare_datafrom src.kpi import (    overall_churn_rate, segment_churn_rate, high_value_churn_ratio,    geographic_risk_index, engagement_drop_indicator, revenue_at_risk)sns.set_style("whitegrid")plt.rcParams["figure.dpi"] = 110pd.set_option("display.max_columns", None)

## 2. Data Ingestion & ValidationLoad the raw dataset, validate binary fields and category labels, remove non-analytical fields, and build all derived segmentation columns.

In [ ]:
df = load_and_prepare_data("../data/European_Bank.csv")print("Shape:", df.shape)df.head()

In [ ]:
df.info()

In [ ]:
# Confirm churn labeling accuracy / class balancedf["Exited"].value_counts(normalize=True).mul(100).round(2)

## 3. Descriptive Statistics

In [ ]:
df.describe()

## 4. Overall Churn Rate (KPI)

In [ ]:
rate = overall_churn_rate(df)print(f"Overall churn rate: {rate}%")fig, ax = plt.subplots(figsize=(5,5))counts = df["Exited"].value_counts().sort_index()ax.pie(counts, labels=["Retained", "Churned"], autopct="%1.1f%%",       colors=["#4C72B0", "#DD8452"], startangle=90)ax.set_title("Overall Customer Churn Distribution")plt.show()

## 5. Churn Distribution by Segment### 5.1 Geography

In [ ]:
geo = segment_churn_rate(df, "Geography")geo

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))sns.barplot(data=geo, x="Geography", y="ChurnRate(%)", hue="Geography", legend=False, palette="viridis", ax=ax)ax.set_title("Churn Rate by Geography")plt.show()

### 5.2 Age Group

In [ ]:
age = segment_churn_rate(df, "AgeGroup")age

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))sns.barplot(data=age, x="AgeGroup", y="ChurnRate(%)", hue="AgeGroup", legend=False, palette="magma", ax=ax)ax.set_title("Churn Rate by Age Group")plt.show()

### 5.3 Credit Score Band

In [ ]:
cs = segment_churn_rate(df, "CreditScoreBand")cs

### 5.4 Tenure Group

In [ ]:
ten = segment_churn_rate(df, "TenureGroup")ten

### 5.5 Balance Segment

In [ ]:
bal = segment_churn_rate(df, "BalanceSegment")bal

## 6. Comparative Demographic Analysis### 6.1 Gender-based churn

In [ ]:
gen = segment_churn_rate(df, "Gender")gen

### 6.2 Geography x Age interaction

In [ ]:
pivot = df.pivot_table(index="Geography", columns="AgeGroup", values="Exited",                       aggfunc="mean", observed=True) * 100fig, ax = plt.subplots(figsize=(7,4))sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax, cbar_kws={"label": "Churn Rate (%)"})ax.set_title("Churn Rate (%): Geography x Age Group")plt.show()

### 6.3 Financial stability (Number of Products, Activity) vs Churn

In [ ]:
prod = segment_churn_rate(df, "NumOfProducts")prod

In [ ]:
eng = engagement_drop_indicator(df)eng

## 7. High-Value Customer Churn Analysis

In [ ]:
hv_ratio = high_value_churn_ratio(df)print(f"High-value customer churn ratio: {hv_ratio}%")hv = segment_churn_rate(df, "IsHighValue")hv

In [ ]:
risk = revenue_at_risk(df)risk

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))sns.kdeplot(data=df, x="Balance", hue="Exited", fill=True, common_norm=False, ax=ax)ax.set_title("Balance Distribution: Churned vs Retained")plt.show()

## 8. Geographic Risk Index (KPI)

In [ ]:
geographic_risk_index(df)

## 9. Correlation Analysis

In [ ]:
num_cols = ["CreditScore","Age","Tenure","Balance","NumOfProducts",            "HasCrCard","IsActiveMember","EstimatedSalary","Exited"]fig, ax = plt.subplots(figsize=(8,6))sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)ax.set_title("Correlation Matrix of Numerical Features")plt.show()

## 10. Key Findings Summary- **Overall churn rate is ~20.4%** — roughly 1 in 5 customers exit.- **Germany has the highest churn rate** among the three markets, despite having about half the customer base of France.- **Customers aged 46-60 churn at a much higher rate** than younger age bands.- **Customers holding 3-4 products churn far more** than those holding 1-2 products, despite being a small segment — a strong early-warning signal.- **Inactive members churn at roughly double the rate of active members.**- **High-value customers (above-median balance and salary) churn at a higher rate (~25%) than standard customers**, meaning the bank is disproportionately losing its most profitable clients.- **Churned customers carry a material balance exposure** — see `revenue_at_risk()` output above for the total deposit balance and salary exposure tied to churned accounts.These findings directly inform the retention strategy recommendations in the accompanying research paper and executive summary.